# Voxelizer & Geodesic Reward Visualization

Tests the voxel distance field output and simulates how the reward lookup works per-drone per-env.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json

VOXEL_DIR = "voxel_output"

with open(f"{VOXEL_DIR}/grid_metadata.json") as f:
    meta = json.load(f)

resolution   = meta["resolution"]                         # 0.05 m
origin       = np.array(meta["origin"])                   # [-5.55, -5.55, -0.5]
grid_shape   = meta["grid_shape"]                         # [223, 222, 220]
goals_world  = [np.array(g) for g in meta["goals_world"]] # list of goal positions
goal_world   = goals_world[0]                              # first (or only) goal
max_geo_dist = meta["max_geodesic_dist_m"]

print("Resolution:", resolution, "m")
print("Origin:", origin)
print("Grid shape (X,Y,Z):", grid_shape)
print(f"Goals ({len(goals_world)}):", [g.tolist() for g in goals_world])
print("Max geodesic dist:", max_geo_dist, "m")

In [ ]:
data           = np.load(f"{VOXEL_DIR}/distance_field.npz")
occupancy      = data["occupancy"]          # bool (X, Y, Z)
# distance_fields is (num_goals, X, Y, Z) — one field per waypoint
dist_fields_all = data["distance_fields"]   # float32 (num_goals, X, Y, Z)
dist_field      = dist_fields_all[0]        # first goal's field (X, Y, Z)

print("Occupancy shape:", occupancy.shape, "  dtype:", occupancy.dtype)
print("Distance fields shape:", dist_fields_all.shape, "  dtype:", dist_fields_all.dtype)
print(f"Occupied voxels: {occupancy.sum():,} / {occupancy.size:,} ({100*occupancy.mean():.1f}%)")
print(f"Dist field[0] — min: {dist_field[~occupancy].min():.2f} m  max: {dist_field[~occupancy].max():.2f} m")

## Helper: world ↔ voxel coordinate conversion

In [ ]:
def world_to_voxel(pos_world):
    """Convert world-local position (relative to env origin) to voxel indices."""
    idx = np.floor((np.array(pos_world) - origin) / resolution).astype(int)
    idx = np.clip(idx, 0, np.array(grid_shape) - 1)
    return idx

def voxel_to_world(idx):
    return np.array(idx) * resolution + origin

def query_geo_dist(pos_world):
    idx = world_to_voxel(pos_world)
    return dist_field[idx[0], idx[1], idx[2]] * resolution  # voxel steps → meters

def euclidean_dist(pos_world):
    return np.linalg.norm(np.array(pos_world) - goal_world)

# Derive goal voxel (not stored in metadata anymore)
goal_voxel = world_to_voxel(goal_world).tolist()

# Goal sanity check
gv = goal_voxel
goal_occ  = occupancy[gv[0], gv[1], gv[2]]
goal_dist = dist_field[gv[0], gv[1], gv[2]]
print(f"=== GOAL SANITY CHECK ===")
print(f"Goal world: {goal_world.tolist()}  → voxel {gv}")
print(f"Occupied: {goal_occ}   dist: {goal_dist:.4f}")
if goal_occ:
    print("  *** GOAL IS INSIDE AN OBSTACLE ***")
elif goal_dist != 0.0:
    print(f"  *** WARNING: goal voxel dist={goal_dist:.4f} (not 0) ***")
else:
    print("  Goal voxel is FREE and dist=0. ✓")

# Sanity checks
spawn_pos = [3.5, 4.0, 5.0]  # drone spawn (env-local)
print()
print(f"Spawn pos {spawn_pos}")
print(f"  → voxel idx:      {world_to_voxel(spawn_pos).tolist()}")
print(f"  → geo dist:       {query_geo_dist(spawn_pos):.2f} m")
print(f"  → euclidean dist: {euclidean_dist(spawn_pos):.2f} m")

## Reward function simulation

In [ ]:
def reward_euclid(pos_world, scale=15.0, step_dt=0.02):
    """Old Euclidean-based reward (kept for comparison only)."""
    d = euclidean_dist(pos_world)
    return (1.0 - np.tanh(d / max_geo_dist)) * scale * step_dt

def reward_geodesic_delta(pos_world, prev_pos_world, scale=15.0, step_dt=0.02):
    """
    New progress-based reward: reward = decrease in geodesic distance per step.
    Mirrors the updated _get_rewards() in obstacle_nav_env.py.
    """
    d_curr = query_geo_dist(pos_world)
    d_prev = query_geo_dist(prev_pos_world)
    return (d_prev - d_curr) * scale * step_dt  # positive = approaching goal

test_positions = [
    ("spawn",         [3.5,  4.0,  5.0]),
    ("near goal",     [-4.3, 3.4,  6.9]),
    ("mid-path",      [0.0,  0.0,  5.0]),
    ("opposite side", [4.5, -4.5,  5.0]),
]

print(f"Geodesic distances from goal (for reference):")
print(f"{'Position':<16} {'Euclid(m)':>10} {'Geodesic(m)':>12}")
print("-" * 42)
for name, pos in test_positions:
    ed = euclidean_dist(pos)
    gd = query_geo_dist(pos)
    print(f"{name:<16} {ed:>10.2f} {gd:>12.2f}")

print()
print("Note: delta reward = (prev_geo_dist - curr_geo_dist) * scale * step_dt")
print("      ~0.01–0.1 m of geodesic progress per step × 15.0 × 0.02 = ~0.003–0.03 reward/step")

## 2D slice visualization: occupancy + distance field at drone flight height

In [ ]:
flight_z_world = 5.0  # meters, typical drone flight height
z_idx = int((flight_z_world - origin[2]) / resolution)
z_idx = np.clip(z_idx, 0, grid_shape[2] - 1)
print(f"Flight height {flight_z_world}m → Z voxel index {z_idx}")

occ_slice  = occupancy[:, :, z_idx]    # (X, Y)
dist_slice = dist_field[:, :, z_idx]   # (X, Y)

# Mask occupied voxels out of distance display
dist_masked = np.where(occ_slice, np.nan, dist_slice)

# Axis tick labels in world coords
x_world = origin[0] + np.arange(grid_shape[0]) * resolution
y_world = origin[1] + np.arange(grid_shape[1]) * resolution

goal_xi = goal_voxel[0]
goal_yi = goal_voxel[1]
spawn_vi = world_to_voxel(spawn_pos)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Occupancy ---
ax = axes[0]
ax.imshow(occ_slice.T, origin="lower", cmap="Greys",
          extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*", ms=14, label="goal")
ax.plot(spawn_pos[0], spawn_pos[1],   "b^", ms=10, label="spawn")
ax.set_title(f"Occupancy slice  z={flight_z_world}m")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend()

# --- Geodesic distance ---
ax = axes[1]
im = ax.imshow(dist_masked.T, origin="lower", cmap="viridis",
               extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.imshow(occ_slice.T, origin="lower", cmap="Greys", alpha=0.3,
          extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*", ms=14, label="goal")
ax.plot(spawn_pos[0], spawn_pos[1],   "b^", ms=10, label="spawn")
plt.colorbar(im, ax=ax, label="Geodesic dist (m)")
ax.set_title(f"Geodesic distance field  z={flight_z_world}m")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend()

plt.tight_layout()
plt.show()

## Euclidean vs Geodesic reward along a straight-line trajectory

In [ ]:
# Simulate a drone flying straight from spawn to goal (ignores walls)
n_steps = 200
t = np.linspace(0, 1, n_steps)
traj = np.array(spawn_pos) + t[:, None] * (goal_world - np.array(spawn_pos))

euclid_dists = np.array([euclidean_dist(p) for p in traj])
geo_dists    = np.array([query_geo_dist(p) for p in traj])
r_euclid     = (1.0 - np.tanh(euclid_dists / 0.8)) * 15.0 * 0.02
r_geodesic   = (1.0 - np.tanh(geo_dists    / 0.8)) * 15.0 * 0.02

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax = axes[0]
ax.plot(t, euclid_dists, label="Euclidean dist", color="steelblue")
ax.plot(t, geo_dists,    label="Geodesic dist",  color="darkorange")
ax.set_ylabel("Distance to goal (m)")
ax.set_title("Straight-line trajectory: Euclidean vs Geodesic distance")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(t, r_euclid,   label="Reward (euclidean)", color="steelblue")
ax.plot(t, r_geodesic, label="Reward (geodesic)",  color="darkorange")
ax.set_xlabel("Trajectory progress (0=spawn, 1=goal)")
ax.set_ylabel("Reward per step")
ax.set_title("Reward comparison along straight-line path")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Geo dist at spawn: {geo_dists[0]:.2f}m   Euclid: {euclid_dists[0]:.2f}m")
print(f"Geo dist at goal:  {geo_dists[-1]:.4f}m  Euclid: {euclid_dists[-1]:.4f}m")

## Vectorized reward lookup — simulate N envs (mirrors training code)

In [ ]:
import torch

device = "cpu"

dist_field_t  = torch.tensor(dist_field,  dtype=torch.float32, device=device)
grid_origin_t = torch.tensor(origin,      dtype=torch.float32, device=device)
grid_res      = float(resolution)
grid_shape_l  = list(dist_field.shape)

def query_distance_field_batch(pos_w, env_origins):
    """Mirrors _query_distance_field in obstacle_nav_env.py."""
    local = pos_w - env_origins
    idx   = ((local - grid_origin_t) / grid_res).long()
    idx[:, 0].clamp_(0, grid_shape_l[0] - 1)
    idx[:, 1].clamp_(0, grid_shape_l[1] - 1)
    idx[:, 2].clamp_(0, grid_shape_l[2] - 1)
    return dist_field_t[idx[:, 0], idx[:, 1], idx[:, 2]] * grid_res  # voxel steps → meters

# Simulate 16 envs, each with a different origin (grid layout, 11m spacing)
N = 16
env_origins = torch.zeros(N, 3, device=device)
env_origins[:, 0] = (torch.arange(N) % 4) * 11.0
env_origins[:, 1] = (torch.arange(N) // 4) * 11.0

# All drones at spawn position (relative to env origin)
spawn_t  = torch.tensor(spawn_pos, dtype=torch.float32, device=device)
pos_w    = env_origins + spawn_t.unsqueeze(0)

geo_dists_batch = query_distance_field_batch(pos_w, env_origins)
print(f"Geo dist at spawn for {N} envs (should all be equal):")
print(geo_dists_batch.numpy())
print(f"Mean: {geo_dists_batch.mean():.4f}  Std: {geo_dists_batch.std():.6f}  (std should be ~0)")

## Vertical profile: geodesic distance vs height at spawn XY

In [ ]:
z_range  = np.linspace(origin[2], origin[2] + grid_shape[2]*resolution - resolution, grid_shape[2])
col_xy   = [spawn_pos[0], spawn_pos[1]]
geo_z    = np.array([query_geo_dist([col_xy[0], col_xy[1], z]) for z in z_range])
occ_z    = np.array([occupancy[world_to_voxel([col_xy[0], col_xy[1], z])[0],
                                world_to_voxel([col_xy[0], col_xy[1], z])[1],
                                world_to_voxel([col_xy[0], col_xy[1], z])[2]] for z in z_range])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_range[~occ_z], geo_z[~occ_z], color="darkorange", lw=1.5, label="geodesic dist (free)")
ax.scatter(z_range[occ_z], [0]*occ_z.sum(), color="gray", s=5, label="occupied", zorder=3)
ax.axvline(spawn_pos[2],  color="blue",  ls="--", label=f"spawn z={spawn_pos[2]}m")
ax.axvline(goal_world[2], color="red",   ls="--", label=f"goal z={goal_world[2]}m")
ax.set_xlabel("Z (m)")
ax.set_ylabel("Geodesic dist to goal (m)")
ax.set_title(f"Geodesic distance vs height at spawn XY ({col_xy})")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## BFS Path Tracing: spawn → goal

Reconstruct the navigable path by greedily descending the distance field (always step to the free 26-connected neighbor with the lowest geodesic distance). This is the exact path the BFS "knows about" — if it renders correctly through the warehouse it confirms the field is valid.

In [ ]:
from itertools import product as iproduct

def trace_bfs_path(start_world, max_steps=5000):
    """
    Greedy gradient descent on dist_field from start to goal.
    At each step: pick the free 26-neighbor with smallest dist value.
    Returns (path_voxels, path_world).
    """
    cur = world_to_voxel(start_world).tolist()
    path_voxels = [cur[:]]
    offsets = [list(o) for o in iproduct([-1,0,1], repeat=3) if any(v != 0 for v in o)]

    for _ in range(max_steps):
        xi, yi, zi = cur
        if dist_field[xi, yi, zi] < resolution:
            break

        best_dist = dist_field[xi, yi, zi]
        best_nb   = None
        for dx, dy, dz in offsets:
            nx, ny, nz = xi+dx, yi+dy, zi+dz
            if not (0 <= nx < grid_shape[0] and 0 <= ny < grid_shape[1] and 0 <= nz < grid_shape[2]):
                continue
            if occupancy[nx, ny, nz]:
                continue
            d = dist_field[nx, ny, nz]
            if d < best_dist:
                best_dist = d
                best_nb   = [nx, ny, nz]

        if best_nb is None:
            print("WARNING: stuck — no improving free neighbor found at", cur)
            break
        cur = best_nb
        path_voxels.append(cur[:])

    path_world = np.array([voxel_to_world(v) for v in path_voxels])
    return path_voxels, path_world

spawn_pos = [3.5, 4.0, 5.0]
path_voxels, path_world = trace_bfs_path(spawn_pos)

print(f"Path length: {len(path_voxels)} voxel steps")
print(f"Path distance (sum of steps): {len(path_voxels) * resolution:.2f} m")
print(f"Geodesic dist at spawn:       {query_geo_dist(spawn_pos):.2f} m")
print(f"Start: {path_world[0]}  →  End: {path_world[-1]}")
print(f"Goal:  {goal_world}")
print(f"Final distance to goal: {np.linalg.norm(path_world[-1] - goal_world):.3f} m")

In [ ]:
x_world = origin[0] + np.arange(grid_shape[0]) * resolution
y_world = origin[1] + np.arange(grid_shape[1]) * resolution
z_world = origin[2] + np.arange(grid_shape[2]) * resolution

# Project occupancy ONLY over the Z range the path actually uses.
# Using any(axis=2) over ALL heights makes floor-level walls appear
# even when the drone flies well above them — that's the "path through obstacles" illusion.
path_z_voxels = [v[2] for v in path_voxels]
z_lo, z_hi = min(path_z_voxels), max(path_z_voxels)
occ_xy = occupancy[:, :, z_lo:z_hi+1].any(axis=2)   # top-down, path height range only
occ_xz = occupancy[:, :, :].any(axis=1)              # side view: collapse Y (full range is fine)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# XY top-down — obstacles at path flight altitude
ax = axes[0]
ax.imshow(occ_xy.T, origin="lower", cmap="Greys", alpha=0.5,
          extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
ax.plot(path_world[:, 0], path_world[:, 1], ".-", color="limegreen", lw=1.5, ms=2, label="BFS path")
ax.plot(path_world[0, 0], path_world[0, 1], "b^", ms=12, zorder=5, label="spawn")
ax.plot(goal_world[0],    goal_world[1],     "r*", ms=14, zorder=5, label="goal")
ax.set_title(f"BFS path — top-down XY\n(obstacles projected over path Z range "
             f"{z_world[z_lo]:.1f}–{z_world[z_hi]:.1f} m)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.legend(); ax.grid(True, alpha=0.2)

# XZ side view
ax = axes[1]
ax.imshow(occ_xz.T, origin="lower", cmap="Greys", alpha=0.5,
          extent=[x_world[0], x_world[-1], z_world[0], z_world[-1]], aspect="equal")
ax.plot(path_world[:, 0], path_world[:, 2], ".-", color="limegreen", lw=1.5, ms=2, label="BFS path")
ax.plot(path_world[0, 0], path_world[0, 2], "b^", ms=12, zorder=5, label="spawn")
ax.plot(goal_world[0],    goal_world[2],     "r*", ms=14, zorder=5, label="goal")
ax.set_title("BFS path — side XZ view")
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)")
ax.legend(); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# Quick collision check: are any path voxels inside occupied cells?
collisions = [v for v in path_voxels if occupancy[v[0], v[1], v[2]]]
print(f"Path voxels in collision: {len(collisions)} (should be 0)")

In [ ]:
import plotly.graph_objects as go
from scipy.ndimage import binary_erosion

# Extract surface voxels: occupied AND has at least one free 6-connected neighbor
# binary_erosion shrinks the solid by 1 voxel — difference = surface shell
interior = binary_erosion(occupancy, border_value=1)
surface  = occupancy & ~interior

surf_idx   = np.argwhere(surface)                       # (N, 3) voxel indices
surf_world = surf_idx * resolution + origin             # world coords

print(f"Total occupied voxels : {occupancy.sum():,}")
print(f"Surface voxels        : {surface.sum():,}  ({100*surface.sum()/occupancy.sum():.1f}% of occupied)")

# Colour surface voxels by height (Z) for depth cue
z_vals  = surf_world[:, 2]
z_norm  = (z_vals - z_vals.min()) / (z_vals.ptp() + 1e-9)

fig = go.Figure()

# Obstacles (surface shell)
fig.add_trace(go.Scatter3d(
    x=surf_world[:, 0], y=surf_world[:, 1], z=surf_world[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=z_norm, colorscale="Greys",
                opacity=0.4, colorbar=dict(title="Z (norm)")),
    name="obstacles",
))

# BFS path
fig.add_trace(go.Scatter3d(
    x=path_world[:, 0], y=path_world[:, 1], z=path_world[:, 2],
    mode="lines",
    line=dict(color=np.linspace(0, 1, len(path_world)), colorscale="Plasma", width=4),
    name="BFS path",
))

# Spawn & goal
fig.add_trace(go.Scatter3d(
    x=[path_world[0, 0]], y=[path_world[0, 1]], z=[path_world[0, 2]],
    mode="markers", marker=dict(size=8, color="blue", symbol="diamond"),
    name="spawn",
))
fig.add_trace(go.Scatter3d(
    x=[goal_world[0]], y=[goal_world[1]], z=[goal_world[2]],
    mode="markers", marker=dict(size=10, color="red", symbol="cross"),
    name="goal",
))

fig.update_layout(
    title="Voxelized warehouse — interactive 3D (drag to rotate, scroll to zoom)",
    scene=dict(
        xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
        aspectmode="data",   # preserves real-world proportions
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(itemsizing="constant"),
)

fig.show()

## Interactive 3D environment viewer (Plotly)

Full voxelized environment with free rotation/zoom. Only surface voxels are shown (occupied voxels with at least one free neighbor) to keep it fast.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa

# --- Subsample occupied voxels for 3D display (full grid is too heavy) ---
occ_idx = np.argwhere(occupancy)  # (N, 3) voxel indices of all obstacles
occ_world = occ_idx * resolution + origin  # world coords

# Only keep voxels within the Z range the path actually uses (+ small margin)
z_lo_w = z_world[z_lo] - 0.5
z_hi_w = z_world[z_hi] + 0.5
mask = (occ_world[:, 2] >= z_lo_w) & (occ_world[:, 2] <= z_hi_w)
occ_world_filt = occ_world[mask]

# Subsample to at most 8000 points so matplotlib stays responsive
rng = np.random.default_rng(42)
if len(occ_world_filt) > 8000:
    idx = rng.choice(len(occ_world_filt), 8000, replace=False)
    occ_world_filt = occ_world_filt[idx]

print(f"Obstacle points shown: {len(occ_world_filt):,}  (Z={z_lo_w:.1f}–{z_hi_w:.1f} m)")

# --- Plot ---
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

# Obstacles — small grey dots
ax.scatter(occ_world_filt[:, 0], occ_world_filt[:, 1], occ_world_filt[:, 2],
           c="grey", s=1, alpha=0.15, label="obstacles")

# Path — colored by progress (blue→red) so you can see the Z changes
n = len(path_world)
colors = plt.cm.plasma(np.linspace(0, 1, n))
for i in range(n - 1):
    ax.plot(path_world[i:i+2, 0], path_world[i:i+2, 1], path_world[i:i+2, 2],
            color=colors[i], lw=1.5)

# Markers
ax.scatter(*path_world[0],  color="blue", s=120, zorder=5, label="spawn")
ax.scatter(*goal_world,     color="red",  s=200, marker="*", zorder=5, label="goal")

ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.set_title("BFS path in 3D\n(path colored blue→red by progress, obstacles in grey)")
ax.legend()
plt.tight_layout()
plt.show()

## 3D path visualization

In [ ]:
# Geodesic distance decreasing monotonically along the path — key sanity check
path_geo_dists = np.array([dist_field[v[0], v[1], v[2]] for v in path_voxels])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(path_geo_dists, color="darkorange", lw=1.5)
ax.set_xlabel("Path step")
ax.set_ylabel("Geodesic distance to goal (m)")
ax.set_title("Geodesic distance along BFS path — must be strictly decreasing")
ax.grid(True, alpha=0.3)

non_decreasing = np.sum(np.diff(path_geo_dists) >= 0)
print(f"Non-decreasing steps: {non_decreasing} / {len(path_geo_dists)-1}  (should be 0)")
print(f"Start dist: {path_geo_dists[0]:.2f} m   End dist: {path_geo_dists[-1]:.4f} m")
plt.tight_layout()
plt.show()

In [ ]:
gv = goal_voxel  # [xi, yi, zi]

# 1. Is the goal voxel itself free?
print("=== Goal voxel check ===")
print(f"Goal voxel {gv}  occupied: {occupancy[gv[0], gv[1], gv[2]]}")
print(f"Goal voxel dist: {dist_field[gv[0], gv[1], gv[2]]:.4f}  (0 = BFS started here, inf = unreachable)")

# 2. Collision check on path
collisions = [v for v in path_voxels if occupancy[v[0], v[1], v[2]]]
print(f"\n=== Path collision check ===")
print(f"Path voxels in collision: {len(collisions)} / {len(path_voxels)}  (must be 0)")

# 3. Vertical occupancy profile at goal XY — shows exactly where the wall is
print(f"\n=== Vertical profile at goal XY ({goal_world[:2]}) ===")
xi, yi = gv[0], gv[1]
occ_col = occupancy[xi, yi, :]
free_ranges = []
in_free = False
for zi, occ in enumerate(occ_col):
    z = origin[2] + zi * resolution
    if not occ and not in_free:
        start = z; in_free = True
    elif occ and in_free:
        free_ranges.append((start, z - resolution)); in_free = False
if in_free:
    free_ranges.append((start, origin[2] + (len(occ_col)-1)*resolution))
print(f"Occupied at goal XY? {occ_col.any()}  |  Free Z ranges: {[(round(a,2), round(b,2)) for a,b in free_ranges]}")
print(f"Goal Z={goal_world[2]}m — in free range? {not occupancy[gv[0], gv[1], gv[2]]}")

# 4. Show occupancy slices at 3 key heights to see where walls are vs. drone path
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
check_heights = [1.0, goal_world[2], spawn_pos[2]]  # floor, goal height, spawn height
for ax, zh in zip(axes, check_heights):
    zi = int((zh - origin[2]) / resolution)
    zi = np.clip(zi, 0, grid_shape[2] - 1)
    sl = occupancy[:, :, zi]
    ax.imshow(sl.T, origin="lower", cmap="Greys",
              extent=[x_world[0], x_world[-1], y_world[0], y_world[-1]], aspect="equal")
    ax.plot(goal_world[0], goal_world[1], "r*", ms=12, label="goal")
    ax.plot(spawn_pos[0],  spawn_pos[1],  "b^", ms=10, label="spawn")
    ax.set_title(f"Occupancy slice  z={zh}m  (zi={zi})\nOccupied: {sl.sum():,}")
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
    ax.legend()
plt.tight_layout()
plt.show()

## Diagnostic: is the goal actually free? Does the path collide?

In [ ]:
# ── Reward parameters (mirrors ObstacleNavEnvCfg) ────────────────────────────
LIN_VEL_SCALE          = -0.05
ANG_VEL_SCALE          = -0.01
DIST_GOAL_SCALE        = 15.0
GOAL_REACHED_BONUS     = 15.0
GOAL_REACHED_THRESHOLD = 0.2   # metres, euclidean
STEP_DT                = 2 * 0.01  # decimation=2, sim_dt=0.01

def compute_reward(pos_world, prev_geo_dist=None, lin_vel=(0., 0., 0.), ang_vel=(0., 0., 0.)):
    """
    Full reward breakdown for one step.
    pos_world    : env-local world position  [x, y, z]
    prev_geo_dist: geodesic distance at the previous step (None → 0 improvement)
    lin_vel      : linear velocity in body frame  (m/s)
    ang_vel      : angular velocity in body frame (rad/s)
    Returns dict with each component + 'total'.

    Reward uses DELTA geodesic distance (progress-based), mirroring _get_rewards().
    """
    pos    = np.array(pos_world, dtype=float)
    lv     = np.array(lin_vel,   dtype=float)
    av     = np.array(ang_vel,   dtype=float)

    geo_dist  = float(query_geo_dist(pos))
    euclid_d  = float(euclidean_dist(pos))

    if prev_geo_dist is None:
        geo_improvement = 0.0  # no progress info
    else:
        prev_clamped = min(prev_geo_dist, geo_dist)  # mirrors torch.minimum guard
        geo_improvement = prev_clamped - geo_dist

    r_lin_vel = np.sum(lv**2) * LIN_VEL_SCALE * STEP_DT
    r_ang_vel = np.sum(av**2) * ANG_VEL_SCALE * STEP_DT
    r_dist    = geo_improvement * DIST_GOAL_SCALE * STEP_DT
    r_reached = GOAL_REACHED_BONUS if euclid_d < GOAL_REACHED_THRESHOLD else 0.0

    total = r_lin_vel + r_ang_vel + r_dist + r_reached
    return {
        "pos"           : pos.tolist(),
        "geo_dist_m"    : geo_dist,
        "geo_improvement": geo_improvement,
        "euclid_dist_m" : euclid_d,
        "r_lin_vel"     : r_lin_vel,
        "r_ang_vel"     : r_ang_vel,
        "r_dist_goal"   : r_dist,
        "r_goal_reached": r_reached,
        "total"         : total,
    }

print("Reward helper ready. step_dt =", STEP_DT, "s")
print("Formula: r_dist = (prev_geo_dist - curr_geo_dist) * 15.0 * 0.02")

---
## Reward Function Scenario Testing

Tests the **full** reward signal from `_get_rewards()` — all four components exactly as in training:

| Component | Formula | Scale |
|-----------|---------|-------|
| `lin_vel` | `−sum(lin_vel_b²) × 0.05 × step_dt` | penalty |
| `ang_vel` | `−sum(ang_vel_b²) × 0.01 × step_dt` | penalty |
| `distance_to_goal` | `(prev_geo_dist − curr_geo_dist) × 15.0 × step_dt` | **progress-based shaping** |
| `goal_reached` | `(euclid_dist < 0.2) × 15.0` | sparse bonus |

`step_dt = decimation × sim_dt = 2 × 0.01 = 0.02 s`

> **Key change from old design:** the shaped reward is now a **delta** (geodesic improvement per step) rather than `1 − tanh(geo_dist / max_dist)`. The old formula returned ≈0 at spawn (agent too far from goal), giving no learning signal. The new formula gives a non-zero reward proportional to progress *wherever* the agent moves.

> **Note:** `goal_reached` uses **euclidean** distance; the shaped reward uses **geodesic** distance.

In [ ]:
EPISODE_LENGTH_S = 10.0   # from ObstacleNavEnvCfg
MAX_STEPS        = int(EPISODE_LENGTH_S / STEP_DT)   # 500 steps

# Resample path_world to MAX_STEPS (drone flies path at constant pace)
t_path = np.linspace(0, 1, len(path_world))
t_sim  = np.linspace(0, 1, MAX_STEPS)
sim_traj = np.stack([np.interp(t_sim, t_path, path_world[:, i]) for i in range(3)], axis=1)

# Estimate velocity from position deltas (finite difference, body ≈ world here)
sim_vel = np.diff(sim_traj, axis=0, prepend=sim_traj[:1]) / STEP_DT  # (N, 3)

# Geodesic distances along trajectory
ep_geo    = np.array([query_geo_dist(p) for p in sim_traj])
ep_euclid = np.array([euclidean_dist(p) for p in sim_traj])

# Delta reward: improvement = prev_geo - curr_geo (clamped on first step)
ep_prev_geo = np.concatenate([[ep_geo[0]], ep_geo[:-1]])  # prev=curr on step 0 → improvement=0
ep_prev_geo_clamped = np.minimum(ep_prev_geo, ep_geo)     # mirrors torch.minimum guard
ep_improvement = ep_prev_geo_clamped - ep_geo              # (N,) positive when approaching

ep_r_dist  = ep_improvement * DIST_GOAL_SCALE * STEP_DT
ep_r_lv    = np.sum(sim_vel**2, axis=1) * LIN_VEL_SCALE * STEP_DT
ep_r_av    = np.zeros(MAX_STEPS)
ep_r_bonus = (ep_euclid < GOAL_REACHED_THRESHOLD).astype(float) * GOAL_REACHED_BONUS
ep_total   = ep_r_dist + ep_r_lv + ep_r_av + ep_r_bonus

ep_cumsum  = np.cumsum(ep_total)
time_axis  = np.arange(MAX_STEPS) * STEP_DT

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

ax = axes[0]
ax.plot(time_axis, ep_r_dist, color="darkorange", label="r_dist_goal (geodesic delta)")
ax.plot(time_axis, ep_r_lv,   color="steelblue",  label="r_lin_vel (penalty)")
ax.fill_between(time_axis, ep_r_bonus, 0, where=ep_r_bonus > 0,
                color="lime", alpha=0.6, label="r_goal_reached")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Reward / step")
ax.set_title("Per-step reward components along optimal BFS path\n(delta reward: positive = geodesic progress)")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(time_axis, ep_total,  color="black", lw=1.5, label="total reward / step")
ax.plot(time_axis, ep_improvement, color="purple", lw=1, ls="--", label="geodesic improvement (m/step)")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_ylabel("Reward / step")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(time_axis, ep_cumsum, color="purple", lw=2, label="cumulative return")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Cumulative return")
ax.set_title("Episode return along optimal path")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Episode length: {MAX_STEPS} steps  ({EPISODE_LENGTH_S} s)")
print(f"Total episode return (optimal BFS path): {ep_total.sum():.3f}")
print(f"  of which r_dist_goal:    {ep_r_dist.sum():.3f}  (≈ total geo dist × {DIST_GOAL_SCALE} × {STEP_DT})")
print(f"           r_lin_vel:      {ep_r_lv.sum():.3f}")
print(f"           r_goal_reached: {ep_r_bonus.sum():.3f}  "
      f"({'fires' if ep_r_bonus.sum() > 0 else 'NEVER FIRES — path too slow'})")
print(f"Total geodesic improvement: {ep_improvement.sum():.2f} m  (should ≈ geo dist at spawn = {ep_geo[0]:.2f} m)")

### Episode-level reward simulation along optimal BFS path

Simulates a full episode where the drone follows the BFS optimal path at a constant speed, computing all reward components at every step. This gives you the theoretical maximum episode return.

In [ ]:
# Approach goal along the BFS path (last N steps)
N_near = min(100, len(path_world))
approach = path_world[-N_near:]

approach_euclid = np.array([euclidean_dist(p) for p in approach])
approach_geo    = np.array([query_geo_dist(p)  for p in approach])

# Delta reward along approach
approach_prev  = np.concatenate([[approach_geo[0]], approach_geo[:-1]])
approach_prev  = np.minimum(approach_prev, approach_geo)  # clamp guard
approach_improvement = approach_prev - approach_geo
approach_r_dist = approach_improvement * DIST_GOAL_SCALE * STEP_DT
approach_bonus  = (approach_euclid < GOAL_REACHED_THRESHOLD).astype(float) * GOAL_REACHED_BONUS
approach_total  = approach_r_dist + approach_bonus

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(approach_euclid, color="steelblue", label="euclidean dist (m)")
ax.plot(approach_geo,    color="darkorange", label="geodesic dist (m)")
ax.axhline(GOAL_REACHED_THRESHOLD, color="lime", ls="--",
           label=f"reached threshold ({GOAL_REACHED_THRESHOLD} m)")
ax.set_ylabel("Distance to goal (m)")
ax.set_title("Final approach to goal along BFS path")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(approach_r_dist,  color="darkorange", lw=2, label="r_dist_goal (geodesic delta)")
ax.plot(approach_improvement, color="purple", lw=1.5, ls="--", label="geodesic improvement (m/step)")
ax.plot(approach_bonus,   color="lime",       lw=2, label=f"r_goal_reached (bonus ×{GOAL_REACHED_BONUS})")
ax.plot(approach_total,   color="black",      lw=1.5, ls="--", label="total (dist + bonus, no vel)")
ax.axvline(np.argmax(approach_euclid < GOAL_REACHED_THRESHOLD),
           color="lime", ls=":", alpha=0.7)
ax.set_xlabel("Step index (0 = N steps before goal)")
ax.set_ylabel("Reward per step")
ax.set_title("Reward components during final approach (delta-based)")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fire_steps = np.where(approach_euclid < GOAL_REACHED_THRESHOLD)[0]
if len(fire_steps):
    print(f"goal_reached fires at step {fire_steps[0]} "
          f"(euclid_dist = {approach_euclid[fire_steps[0]]:.3f} m < {GOAL_REACHED_THRESHOLD} m)")
    print(f"At that step: r_dist_goal = {approach_r_dist[fire_steps[0]]:.5f},  "
          f"bonus = {approach_bonus[fire_steps[0]]:.2f}")
else:
    print("goal_reached never fires in the last N path steps — increase N_near")

### Goal-reached threshold analysis

Sweep euclidean distance from goal. Shows the sharp transition of the sparse bonus and how it relates to the shaped reward.

In [ ]:
speeds     = np.linspace(0, 5, 200)   # m/s or rad/s along a single axis
penalty_lv = speeds**2 * abs(LIN_VEL_SCALE) * STEP_DT   # lin_vel penalty magnitude
penalty_av = speeds**2 * abs(ANG_VEL_SCALE) * STEP_DT   # ang_vel penalty magnitude

# For delta reward: the reward per step = geodesic_improvement * scale * step_dt
# Typical improvement depends on speed: at speed v, improvement ≈ v * step_dt (if moving optimally)
# So r_dist ≈ v * step_dt * DIST_GOAL_SCALE * step_dt = v * 15.0 * 0.02^2 = v * 0.006 per step
#
# Break-even: v * 0.006 = v^2 * |LIN_VEL_SCALE| * step_dt
#           → v_break_even = 0.006 / (0.05 * 0.02) = 6 m/s
v_break_even = (DIST_GOAL_SCALE * STEP_DT**2) / (abs(LIN_VEL_SCALE) * STEP_DT)
r_dist_vs_speed = speeds * STEP_DT * DIST_GOAL_SCALE * STEP_DT  # approx optimal delta reward at speed v

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(speeds, penalty_lv,       label="|r_lin_vel| penalty",           color="steelblue", lw=2)
ax.plot(speeds, penalty_av,       label="|r_ang_vel| penalty (one axis)", color="tomato",    lw=2)
ax.plot(speeds, r_dist_vs_speed,  label="approx r_dist (optimal progress)", color="darkorange", lw=2, ls="--")
ax.axvline(v_break_even, color="grey", ls=":", lw=1.5, label=f"break-even ≈{v_break_even:.1f} m/s")
ax.set_xlabel("Speed (m/s)")
ax.set_ylabel("Reward magnitude (per step)")
ax.set_title("Velocity penalty vs. delta distance reward\n(delta reward ∝ speed × step_dt²)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
print(f"Break-even speed (lin): {v_break_even:.2f} m/s  "
      f"(moving faster than this costs more than it earns)")

# Stacked bar: reward components for increasing speed at spawn
# With delta reward we need a "prev" position; approximate as one step back along BFS path
speed_samples = [0, 0.5, 1.0, 2.0, 3.0, 5.0]
components = {}
for s in speed_samples:
    # Approximate prev geo dist: the drone moved 1 step at speed s from spawn
    prev_pos = np.array(spawn_pos) - np.array([s * STEP_DT, 0, 0])  # moved in +X direction
    prev_geo = float(query_geo_dist(prev_pos))
    r = compute_reward(spawn_pos, prev_geo_dist=prev_geo, lin_vel=[s, 0, 0], ang_vel=[0, 0, 0])
    components.setdefault("r_dist_goal",  []).append(r["r_dist_goal"])
    components.setdefault("r_lin_vel",    []).append(r["r_lin_vel"])
    components.setdefault("r_ang_vel",    []).append(r["r_ang_vel"])

ax = axes[1]
bar_colors = {"r_dist_goal": "limegreen", "r_lin_vel": "steelblue", "r_ang_vel": "tomato"}
bottoms = np.zeros(len(speed_samples))
for key in ["r_dist_goal", "r_lin_vel", "r_ang_vel"]:
    vals = np.array(components[key])
    ax.bar(range(len(speed_samples)), vals, bottom=bottoms, label=key, color=bar_colors[key], alpha=0.8)
    bottoms += vals
ax.set_xticks(range(len(speed_samples)))
ax.set_xticklabels([f"{s} m/s" for s in speed_samples])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Reward per step")
ax.set_title("Reward components at spawn vs. forward speed\n(delta reward, prev pos = 1 step back in +X)")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

### Velocity penalty analysis

How much does speed cost relative to the distance-shaping gain?

In [ ]:
from matplotlib.patches import Circle

viz_z = goal_world[2]  # visualise at goal height
z_vi  = int((viz_z - origin[2]) / resolution)
z_vi  = np.clip(z_vi, 0, grid_shape[2] - 1)

occ_slice_viz  = occupancy[:, :, z_vi]          # (X, Y)
dist_slice_viz = dist_field[:, :, z_vi]          # (X, Y) in voxel steps

dist_m_viz = dist_slice_viz * resolution         # convert to meters
dist_m_viz_masked = np.where(occ_slice_viz, np.nan, dist_m_viz)

# Gradient magnitude of geodesic distance field — represents "reward rate per meter of movement"
# The gradient is ≈1 everywhere along navigable paths (BFS descent), higher at bottlenecks/detours
gx, gy = np.gradient(dist_m_viz, resolution)
grad_magnitude = np.sqrt(gx**2 + gy**2)
grad_magnitude = np.where(occ_slice_viz, np.nan, grad_magnitude)

# Old tanh-based reward map (for comparison)
r_old_map = (1.0 - np.tanh(dist_m_viz / max_geo_dist)) * DIST_GOAL_SCALE * STEP_DT
r_old_map = np.where(occ_slice_viz, np.nan, r_old_map)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
ext = [x_world[0], x_world[-1], y_world[0], y_world[-1]]

# --- Geodesic distance field (what agent descends) ---
ax = axes[0]
im = ax.imshow(dist_m_viz_masked.T, origin="lower", cmap="viridis_r",
               extent=ext, aspect="equal")
ax.imshow(occ_slice_viz.T, origin="lower", cmap="Greys", alpha=0.35, extent=ext, aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*",  ms=14, zorder=5, label="goal")
ax.plot(spawn_pos[0],  spawn_pos[1],  "b^",  ms=10, zorder=5, label="spawn")
ax.add_patch(Circle((goal_world[0], goal_world[1]), GOAL_REACHED_THRESHOLD,
                     fill=False, ec="lime", lw=2, ls="--"))
plt.colorbar(im, ax=ax, label="geodesic dist (m)")
ax.set_title(f"Geodesic distance field\n(z={viz_z}m, agent descends toward 0)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.legend(fontsize=8)

# --- Gradient magnitude (reward per unit displacement) ---
ax = axes[1]
im = ax.imshow(grad_magnitude.T, origin="lower", cmap="plasma",
               vmin=0, vmax=3, extent=ext, aspect="equal")
ax.imshow(occ_slice_viz.T, origin="lower", cmap="Greys", alpha=0.35, extent=ext, aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*",  ms=14, zorder=5, label="goal")
ax.plot(spawn_pos[0],  spawn_pos[1],  "b^",  ms=10, zorder=5, label="spawn")
plt.colorbar(im, ax=ax, label="|∇geo_dist| (m/m)")
ax.set_title(f"Geodesic gradient magnitude\n(≈1 = clear path, >1 = detour zone, 0 = flat/stuck)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.legend(fontsize=8)

# --- Old tanh-based reward (kept for comparison) ---
ax = axes[2]
im = ax.imshow(r_old_map.T, origin="lower", cmap="plasma",
               vmin=0, vmax=DIST_GOAL_SCALE * STEP_DT, extent=ext, aspect="equal")
ax.imshow(occ_slice_viz.T, origin="lower", cmap="Greys", alpha=0.35, extent=ext, aspect="equal")
ax.plot(goal_world[0], goal_world[1], "r*",  ms=14, zorder=5, label="goal")
ax.plot(spawn_pos[0],  spawn_pos[1],  "b^",  ms=10, zorder=5, label="spawn")
ax.add_patch(Circle((goal_world[0], goal_world[1]), GOAL_REACHED_THRESHOLD,
                     fill=False, ec="lime", lw=2, ls="--"))
plt.colorbar(im, ax=ax, label="reward / step (OLD)")
ax.set_title(f"OLD reward (1 − tanh(geo/max))\n≈ 0 at spawn — this caused the learning failure")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.legend(fontsize=8)

plt.suptitle("Delta reward: agent earns reward for geodesic progress per step\n"
             "(gradient ≈1 → consistent signal everywhere; old tanh → zero at spawn)", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

print(f"Geodesic dist at spawn: {query_geo_dist(spawn_pos):.2f} m")
print(f"Old reward at spawn:    {(1.0 - np.tanh(query_geo_dist(spawn_pos)/max_geo_dist)) * DIST_GOAL_SCALE * STEP_DT:.5f}  ← was ~0")
print(f"New reward at spawn (0.1m/step progress): {0.1 * DIST_GOAL_SCALE * STEP_DT:.4f}  ← non-zero wherever agent moves")

### 2D reward heatmap at goal height (zero velocity)

Shows where the reward shaping pushes the drone. `r_goal_reached` is not shown here (it only fires within 0.2 m).

---
## Reward Function Scenario Testing

Tests the **full** reward signal from `_get_rewards()` — all four components exactly as in training:

| Component | Formula | Scale |
|-----------|---------|-------|
| `lin_vel` | `−sum(lin_vel_b²) × 0.05 × step_dt` | penalty |
| `ang_vel` | `−sum(ang_vel_b²) × 0.01 × step_dt` | penalty |
| `distance_to_goal` | `(prev_geo_dist − curr_geo_dist) × 15.0 × step_dt` | **progress-based shaping** |
| `goal_reached` | `(euclid_dist < 0.2) × 15.0` | sparse bonus |

`step_dt = decimation × sim_dt = 2 × 0.01 = 0.02 s`

> **Key change from old design:** the shaped reward is now a **delta** (geodesic improvement per step) rather than `1 − tanh(geo_dist / max_dist)`. The old formula returned ≈0 at spawn, giving no learning signal. The new formula gives non-zero reward proportional to progress wherever the agent moves.

> **Note:** `goal_reached` uses **euclidean** distance; the shaped reward uses **geodesic** distance.